# Insurance Pricing with pymgcv

This notebook demonstrates a **production actuarial workflow** using Tweedie GAMs
for insurance loss-cost modelling — the most common GAM use case in the insurance industry.

## What You'll Learn
1. Building a Tweedie GLM/GAM for zero-inflated loss-cost data
2. Using `offset()` for exposure-adjusted rates
3. Mixing smooth and parametric (factor) terms
4. Over-smoothing with `gamma=1.2` for pricing stability
5. Visualizing smooth relativities
6. Exporting model output for pricing systems

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from pymgcv import GAM, Tweedie, plot_smooth, aic

np.random.seed(42)

## 1. Load Insurance Data

5,000 synthetic insurance policies with:
- **Response:** `capped_claims_kusd` — claims in thousands (zero-inflated, right-skewed → Tweedie)
- **Offset:** `log_duration` — log policy duration (converts model to loss rate)
- **Smooth terms:** `vehicle_age`, `driver_age`, `bonus_malus`, `log10_exposure_usd`, `annual_mileage_km`
- **Parametric terms:** `class_B/C/D` (vehicle class), `is_urban/is_suburban` (region)

In [ ]:
df = pd.read_csv("../insurance_loss_cost_data.csv")
print(f"Shape: {df.shape}")
print(f"Zero claims: {(df['capped_claims_kusd'] == 0).mean():.1%}")
df.describe().round(2)

## 2. Fit the Tweedie GAM

Key choices:
- **`Tweedie(power=1.5)`** — Compound Poisson-Gamma, standard for insurance data
- **`gamma=1.2`** — Mild over-smoothing to prevent overfitting (common in pricing)
- **`offset(log_duration)`** — Makes the model predict loss *rate* per unit duration
- **REML** — Restricted maximum likelihood for smoothing parameter selection

In [ ]:
formula = (
    "capped_claims_kusd ~ "
    "offset(log_duration) + "
    "s(vehicle_age, k=8) + "
    "s(driver_age, k=10) + "
    "s(bonus_malus, k=8) + "
    "s(log10_exposure_usd, k=6) + "
    "s(annual_mileage_km, k=6) + "
    "class_B + class_C + class_D + "
    "is_urban + is_suburban"
)

model = GAM(
    formula=formula,
    data=df,
    family=Tweedie(power=1.5),
    method="REML",
    gamma=1.2,
    control={"maxit": 150, "epsilon": 1e-5},
)
model.fit(verbose=False)
print(model.summary())

## 3. Smooth Term Relativities

Plot the partial effects of each rating factor. These are the *relativities* that actuaries
use to build pricing tariffs.

In [ ]:
smooth_vars = ["vehicle_age", "driver_age", "bonus_malus",
               "log10_exposure_usd", "annual_mileage_km"]

fig, axes = plt.subplots(2, 3, figsize=(16, 10))
for i, var in enumerate(smooth_vars):
    plot_smooth(model, var_name=var, ax=axes.ravel()[i], ci=0.95)
    axes.ravel()[i].set_title(f"s({var})", fontweight="bold")
axes.ravel()[5].set_visible(False)
fig.suptitle("Insurance Loss Cost — Smooth Relativities", fontsize=14, fontweight="bold")
plt.tight_layout()
plt.show()

## 4. Parametric Coefficients

The parametric terms give **multiplicative relativities** (exp(β)) for categorical risk factors.

In [ ]:
# Extract parametric coefficients
param_names = ["(Intercept)", "class_B", "class_C", "class_D", "is_urban", "is_suburban"]
betas = model.beta
mm = model.model_matrix

for name in param_names:
    if name in mm.column_names:
        idx = mm.column_names.index(name)
        coef = betas[idx]
        print(f"{name:20s}  β = {coef:+.4f}  exp(β) = {np.exp(coef):.4f}  ({(np.exp(coef)-1)*100:+.1f}%)")

## 5. Predictions and Balance Ratio

The **balance ratio** (total predicted / total actual) should be close to 1.0
for a well-calibrated model.

In [ ]:
df["predicted"] = model.predict(df, scale="response")

balance = df["predicted"].sum() / df["capped_claims_kusd"].sum()
print(f"Balance ratio: {balance:.4f}")
print(f"AIC: {aic(model):.1f}")
print(f"Mean actual: {df['capped_claims_kusd'].mean():.4f}")
print(f"Mean predicted: {df['predicted'].mean():.4f}")

## 6. EDF and Smoothing Parameters

Effective degrees of freedom (EDF) indicate smooth complexity:
- EDF ≈ 1 → nearly linear
- EDF close to k → highly nonlinear (may need larger k)

In [ ]:
for term, info in model.edf_per_smooth.items():
    print(f"{term:45s}  EDF = {info['edf']:.3f}")

## 7. R Equivalent

```r
library(mgcv)
df <- read.csv("insurance_loss_cost_data.csv")

model <- gam(
  capped_claims_kusd ~ offset(log_duration) +
    s(vehicle_age, k=8) + s(driver_age, k=10) + s(bonus_malus, k=8) +
    s(log10_exposure_usd, k=6) + s(annual_mileage_km, k=6) +
    class_B + class_C + class_D + is_urban + is_suburban,
  data    = df,
  family  = Tweedie(p=1.5, link="log"),
  method  = "REML",
  gamma   = 1.2,
  control = gam.control(maxit=150, epsilon=1e-5)
)
summary(model)
```

With fixed `p=1.5`, all pymgcv outputs match R mgcv within **< 0.5%**.